# Pairs Trading with Mean-Reverting Stochastic Processes
**MAC F244 project (Topic 17)**: Ornstein-Uhlenbeck spreads, ADF/Engle-Granger pair selection and optimal trading bands, on US and Indian stocks (2010-2026).

**How to use this notebook (temporary master copy)**
- Every section is tagged with its owner (M1-M5). Edit only your own sections, then tick your task in `Project_Tracker.csv`.
- The maths and the backtest live in `src/`; this notebook calls them and makes every figure and table in the report.
- Figures are saved to `figs/` as PNG and EPS with `savefig(fig, name)`, using the names `report/main.tex` expects.
- Cells marked **HEAVY** run the full walk-forward backtests (slow the first time, cached in `results/cache/` afterwards). They only run when `RUN_HEAVY = True`.
- Before submission, M1 runs *Restart & Run All* once and commits the outputs.

| Section | Content | Owner |
| --- | --- | --- |
| 1 | Setup | M1 |
| 2 | Data | M2 |
| 3 | OU model and simulation | M3 |
| 4 | E1 estimator accuracy, E2 ADF power and size | M3 |
| 5 | Pair selection on one window | M2 |
| 6 | Trading rules, Bertram bands, E3 | M4 |
| 7 | Backtest engine checks, E4 synthetic pair | M1 |
| 8 | E5 headline backtest | M1, M4 |
| 9 | E6 transaction costs | M5 |
| 10 | E7 parameter sensitivity | M5 |
| 11 | E8 markets, sectors, sub-periods | M5, M2 |
| 12 | E9 cointegration persistence | M2 |
| 13 | E10 Kalman hedge ratio (stretch) | M4 |
| 14 | Conclusions | M5 |

## 1. Setup  (Owner: M1)
Imports, the fixed seed (42) and the switch for the heavy backtests. All parameters live in `src/config.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import load_prices, benchmark_returns, sectors
from src.ou import (simulate_ou, fit_ou, bertram_cycle_time, bertram_bands,
                    bertram_return_rate, first_passage_cycle_mc)
from src.cointegration import adf_test, engle_granger, engle_granger_both, naive_residual_adf
from src.selection import select_cointegrated, select_distance
from src.backtest import run_selection, simulate, pair_returns
from src.metrics import compute_metrics, metrics_table
from src.plots import savefig, COLORS

np.random.seed(C.SEED)
rng = np.random.default_rng(C.SEED)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

RUN_HEAVY = False   # True runs the full walk-forward backtests (sections 8-13)
STRATS = ['distance', 'zscore', 'sscore', 'bertram']

## 2. Data  (Owner: M2)
Daily adjusted closes from Yahoo Finance (`yfinance`), cached in `data/raw/` (see `data/raw/README.md` for retrieval date and flagged jumps). 56 US stocks in 7 sectors plus EWA/EWC, and 26 NSE stocks in 5 sectors. Pairs are formed only within a sector.

*M2: add your 5-line notes on Gatev et al. (2006) and Engle & Granger (1987) here (task T04).*

In [ ]:
px_us = load_prices('us')
px_in = load_prices('india')
bench_us = benchmark_returns('us')
bench_in = benchmark_returns('india')

summary = pd.DataFrame({
    m: {'tickers': p.shape[1], 'first day': p.index[0].date(), 'last day': p.index[-1].date(),
        'trading days': len(p), 'missing %': round(100 * p.isna().mean().mean(), 2)}
    for m, p in {'US': px_us, 'India': px_in}.items()}).T
summary

In [ ]:
# Two classic pairs, prices normalised to 1 at the start
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, (a, b) in zip(axes, [('KO', 'PEP'), ('XOM', 'CVX')]):
    p = px_us[[a, b]].dropna()
    (p / p.iloc[0]).plot(ax=ax, lw=1)
    ax.set_title(f'{a} vs {b}'); ax.set_ylabel('Normalised price'); ax.set_xlabel('')
savefig(fig, 'fig_data_pairs')

## 3. The Ornstein-Uhlenbeck model  (Owner: M3)
The log-price spread $X_t = \ln P^A_t - \beta \ln P^B_t - \mu$ is modelled as
$$dX_t = \kappa(\theta - X_t)\,dt + \sigma\,dW_t, \qquad \kappa > 0.$$
Itô's lemma on $e^{\kappa t}X_t$ gives
$$X_t = \theta + (X_0-\theta)e^{-\kappa t} + \sigma\int_0^t e^{-\kappa(t-s)}\,dW_s,$$
so $X_t$ is Gaussian with mean $\theta + (X_0-\theta)e^{-\kappa t}$, variance $\sigma^2(1-e^{-2\kappa t})/(2\kappa)$, stationary law $N(\theta, \sigma^2/2\kappa)$ and half-life $\ln 2/\kappa$.

**Exact discretisation.** With step $\Delta = 1/252$:
$$X_{t+\Delta} = a + bX_t + \varepsilon_t,\quad b = e^{-\kappa\Delta},\quad a = \theta(1-b),\quad \mathrm{Var}(\varepsilon_t) = \sigma^2\frac{1-e^{-2\kappa\Delta}}{2\kappa}.$$
OLS of $X_{t+\Delta}$ on $X_t$ is the conditional MLE, and $\hat\kappa = -\ln\hat b/\Delta$, $\hat\theta = \hat a/(1-\hat b)$, $\sigma_{eq} = \hat s_\varepsilon/\sqrt{1-\hat b^2}$.

**Euler-Maruyama** uses $X_{t+\Delta} = X_t + \kappa(\theta - X_t)\Delta + \sigma\sqrt{\Delta}Z$, i.e. $b \approx 1-\kappa\Delta$. The noise is additive, so Milstein's correction term is zero and Milstein equals Euler-Maruyama.

*M3: check every line against report section 2 (task T05).*

In [ ]:
# Exact vs Euler-Maruyama paths driven by the same shocks (same seed)
kappa, theta, sigma, n = 25.0, 0.0, 0.2, 252
x_exact = simulate_ou(kappa, theta, sigma, 0.08, n, method='exact', rng=np.random.default_rng(1))[0]
x_euler = simulate_ou(kappa, theta, sigma, 0.08, n, method='euler', rng=np.random.default_rng(1))[0]
sig_eq = sigma / np.sqrt(2 * kappa)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(x_exact, lw=1, label='Exact scheme')
ax.plot(x_euler, lw=1, ls='--', label='Euler-Maruyama')
ax.axhspan(theta - sig_eq, theta + sig_eq, color='grey', alpha=0.15, label=r'$\theta \pm \sigma_{eq}$')
ax.set_xlabel('Trading day'); ax.set_ylabel('Spread'); ax.legend()
savefig(fig, 'fig_ou_paths')

# T11: the estimator recovers known parameters on a long exact path
for k in [5, 25, 50]:
    x = simulate_ou(k, 0.0, 0.2, 'stationary', 252 * 40, rng=rng)[0]
    p = fit_ou(x)
    print(f'kappa={k:>3}: kappa_hat={p.kappa:6.2f}, theta_hat={p.theta:+.4f}, sigma_hat={p.sigma:.4f}, half-life={p.half_life:5.1f} d')

## 4. E1 and E2: can we estimate and detect mean reversion?  (Owner: M3)
**E1.** Monte Carlo of $\hat\kappa$ for windows of 63-504 days. Yu (2012) shows $\hat\kappa$ is biased upward in short samples, which makes half-lives look shorter than they are. We also compare the exact estimator with the Euler estimator $(1-\hat b)/\Delta$.

**E2.** The ADF regression $\Delta X_t = \alpha + \gamma X_{t-1} + \sum_i\delta_i\Delta X_{t-i} + \varepsilon_t$ has $\gamma = e^{-\kappa\Delta}-1$, so $H_0:\gamma=0$ is $H_0:\kappa=0$. We measure (a) the power of ADF across half-life and window length, which justifies the 12-month formation window, and (b) the size of the residual test on two independent random walks: plain ADF critical values vs Engle-Granger (MacKinnon 2010) critical values (spurious regression, Granger & Newbold 1974).

In [ ]:
# E1: bias and RMSE of kappa_hat
kappas, lengths, n_paths = [5, 10, 25, 50], [63, 126, 252, 504], 1000
rows = []
for k in kappas:
    for T in lengths:
        x = simulate_ou(k, 0.0, 0.2, 'stationary', T, n_paths=n_paths, rng=rng)
        k_exact = np.array([fit_ou(p).kappa for p in x])
        k_euler = np.array([fit_ou(p, estimator='euler').kappa for p in x])
        rows.append(dict(kappa=k, days=T, valid_share=np.isfinite(k_exact).mean(),
                         rel_bias_exact=np.nanmean(k_exact) / k - 1,
                         rel_rmse_exact=np.sqrt(np.nanmean((k_exact - k) ** 2)) / k,
                         rel_bias_euler=np.nanmean(k_euler) / k - 1))
e1 = pd.DataFrame(rows)
e1.to_csv('results/e1_ou_estimation.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 3.6))
for k in kappas:
    d = e1[e1.kappa == k]
    ax.plot(d.days, 100 * d.rel_bias_exact, marker='o', label=f'kappa = {k} / yr')
ax.set_xscale('log'); ax.set_xticks(lengths); ax.set_xticklabels(lengths)
ax.set_xlabel('Window length (trading days)'); ax.set_ylabel('Relative bias of kappa_hat (%)'); ax.legend()
savefig(fig, 'fig_e1_ou_estimation')
e1

In [ ]:
# E2a: ADF power over half-life x window length (rejection rate at 5%)
half_lives, lengths, reps = [5, 10, 20, 40, 80], [63, 126, 252, 504], 300
power = pd.DataFrame(index=half_lives, columns=lengths, dtype=float)
for hl in half_lives:
    k = np.log(2) / (hl * C.DT)
    for T in lengths:
        x = simulate_ou(k, 0.0, 0.2, 'stationary', T, n_paths=reps, rng=rng)
        power.loc[hl, T] = np.mean([adf_test(p, maxlag=5)[1] < 0.05 for p in x])
power.to_csv('results/e2_adf_power.csv')

fig, ax = plt.subplots(figsize=(5.5, 3.6))
im = ax.imshow(power.values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(lengths)), lengths); ax.set_yticks(range(len(half_lives)), half_lives)
ax.set_xlabel('Window length (trading days)'); ax.set_ylabel('True half-life (days)')
for i in range(len(half_lives)):
    for j in range(len(lengths)):
        ax.text(j, i, f'{power.values[i, j]:.2f}', ha='center', va='center', color='w', fontsize=8)
fig.colorbar(im, label='ADF rejection rate')
ax.grid(False)
savefig(fig, 'fig_e2_adf_power')
power

In [ ]:
# E2b: size of the residual test when the truth is 'not cointegrated'
reps, n = 1000, 252
naive, eg = [], []
for _ in range(reps):
    y, x = np.cumsum(rng.standard_normal((2, n)) * 0.01, axis=1)
    naive.append(naive_residual_adf(y, x)[1] < 0.05)
    eg.append(engle_granger(y, x).pvalue < 0.05)
e2_size = pd.Series({'Plain ADF critical values on residuals': np.mean(naive),
                     'Engle-Granger (MacKinnon) critical values': np.mean(eg)},
                    name='Rejection rate of a nominal 5% test')
e2_size.to_csv('results/e2_size.csv')
e2_size

*M3: interpret E1 and E2 here in 3-4 sentences (bias size at 252 days, which half-lives a 12-month window can detect, how badly the naive test over-rejects).*

## 5. Pair selection on one formation window  (Owner: M2)
Engle-Granger in both directions (keep a pair only if both p-values are below 5%), positive hedge ratio, half-life 2-63 trading days, at least 12 mean crossings a year (Sarmento & Horta 2020), ranked by the EG statistic, top 20. Benjamini-Hochberg (q = 0.10) is reported as a diagnostic. These numbers go into the proposal's *progress so far* paragraph (task T13).

In [ ]:
form = np.log(px_us.iloc[-C.FORMATION_DAYS:]).dropna(axis=1)
print(f'Formation window: {form.index[0].date()} to {form.index[-1].date()}')
chosen, diag, tested = select_cointegrated(form, sectors('us'))
print(diag)
top = pd.DataFrame([dict(pair=f'{s.a}/{s.b}', sector=s.sector, beta=s.beta, eg_stat=s.eg_stat,
                         eg_p=s.eg_pvalue, half_life_days=s.half_life, crossings_per_year=s.crossings)
                    for s in chosen])
top.head(10)

## 6. Trading rules and Bertram's optimal bands  (Owner: M4)
- **S0 distance** (Gatev et al. 2006): open when normalised prices diverge by 2 formation standard deviations, close at the crossing.
- **S1 z-score**: open at $|z|>2$, close at $|z|<0.5$.
- **S2 s-score** (Avellaneda & Lee 2010): $s = (X-\hat\theta)/\sigma_{eq}$; open long $s<-1.25$, open short $s>1.25$, close long $s>-0.50$, close short $s<0.75$.
- **S3 Bertram (2010)**: for the centred OU with entry $a$, exit $m$ and round-trip cost $c$,
$$\mathbb{E}[T] = \frac{\pi}{\kappa}\left[\mathrm{erfi}\left(\frac{m\sqrt\kappa}{\sigma}\right) - \mathrm{erfi}\left(\frac{a\sqrt\kappa}{\sigma}\right)\right],\qquad \mu(a,m) = \frac{m-a-c}{\mathbb{E}[T]},$$
from the backward equation $\tfrac12\sigma^2u'' - \kappa x u' = -1$. With $a=-m$ we maximise $\mu$ numerically.

*M4: add your 5-line notes on Avellaneda & Lee and Bertram here (task T06).*

In [ ]:
# Proposal figure (T12): spread, OU fit and s-score bands on the latest year
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, (a, b) in zip(axes, [('KO', 'PEP'), ('EWA', 'EWC')]):
    lp = np.log(px_us[[a, b]].iloc[-C.FORMATION_DAYS:].dropna())
    best, p_both = engle_granger_both(lp[a].values, lp[b].values, a, b)
    ou = fit_ou(best.resid)
    s = (best.resid - ou.theta) / ou.sigma_eq if ou.valid else best.resid / best.resid.std()
    ax.plot(lp.index, s, lw=1)
    for lvl, ls in [(1.25, '-'), (-1.25, '-'), (0.75, ':'), (-0.5, ':')]:
        ax.axhline(lvl, color='grey', ls=ls, lw=0.8)
    hl = f'{ou.half_life:.0f} d' if ou.valid else 'n/a'
    ax.set_title(f'{best.y} - {best.beta:.2f} {best.x}: EG p = {p_both:.2f}, half-life {hl}')
    ax.set_ylabel('s-score')
savefig(fig, 'fig_proposal_spread')

In [ ]:
# E3: Bertram's E[T] against simulated first-passage cycles, and m* against cost
kappa, sigma = 20.0, 0.1
sig_eq = sigma / np.sqrt(2 * kappa)
m_grid = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0]) * sig_eq
formula = bertram_cycle_time(-m_grid, m_grid, kappa, sigma) * 252
mc = [first_passage_cycle_mc(kappa, sigma, m, n_steps=1_000_000, rng=rng).mean() * 252 for m in m_grid]
e3 = pd.DataFrame({'m / sigma_eq': m_grid / sig_eq, 'E[T] formula (days)': formula, 'E[T] Monte Carlo (days)': mc})
e3.to_csv('results/e3_bertram_check.csv', index=False)

costs = np.linspace(0, 1.0, 21) * sig_eq
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(e3.iloc[:, 0], formula, label='Formula')
axes[0].plot(e3.iloc[:, 0], mc, 'o', label='Monte Carlo')
axes[0].set_xlabel('Band half-width m / sigma_eq'); axes[0].set_ylabel('Expected cycle (days)'); axes[0].legend()
for k in [5, 20, 50]:
    sigma_k = sig_eq * np.sqrt(2 * k)   # same sigma_eq, different speed
    m_star = [bertram_bands(k, sigma_k, c) / sig_eq for c in costs]
    axes[1].plot(costs / sig_eq, m_star, label=f'kappa = {k}')
axes[1].set_xlabel('Round-trip cost / sigma_eq'); axes[1].set_ylabel('Optimal m* / sigma_eq'); axes[1].legend()
savefig(fig, 'fig_e3_bertram')
e3

## 7. Backtest engine checks and E4 synthetic pair  (Owner: M1)
Signal at the close of day $t$, executed at the close of $t+1$; parameters frozen at the end of formation; positions closed on the last trading day. A pair's daily return on gross capital is $\text{pos}\cdot(r^A - \beta r^B)/(1+|\beta|)$ minus costs per leg per transaction.

**E4** feeds the whole pipeline a synthetic pair with a known OU spread, where Bertram's formula predicts the return.

In [ ]:
# Look-ahead check (T24): a signal that turns on at day 10 must be held from day 11
desired = np.zeros(30); desired[10:20] = 1
ra = np.full(30, 0.01); rb = np.zeros(30)
net, held = pair_returns(desired, 1.0, ra, rb, cost=0.0)
assert held[10] == 0 and held[11] == 1, 'execution lag broken'
assert net[11] == 0 and net[12] > 0, 'return earned before the position was held'
print('Look-ahead check passed')

In [ ]:
# E4: synthetic cointegrated pair: log B is a random walk, A = 0.1 + 1.2 log B + OU spread
n_days, kappa_true, sig_eq_true, beta_true = 252 * 12, 25.0, 0.03, 1.2
sigma_true = sig_eq_true * np.sqrt(2 * kappa_true)
g = np.random.default_rng(7)
log_b = np.log(50) + np.cumsum(0.0003 + 0.015 * g.standard_normal(n_days))
spread = simulate_ou(kappa_true, 0.0, sigma_true, 0.0, n_days - 1, rng=g)[0]
log_a = 0.1 + beta_true * log_b + spread
dates = pd.bdate_range('2010-01-01', periods=n_days)
px_syn = pd.DataFrame({'A': np.exp(log_a), 'B': np.exp(log_b)}, index=dates)

sel_syn = run_selection('synthetic', px_syn, {'Synthetic': ['A', 'B']}, k=1, use_cache=False)
cost_bps = 5
res_syn = {s: simulate(s, px_syn, sel_syn, cost_bps, k=1) for s in ['sscore', 'bertram']}
realised = {s: r.committed.mean() * 252 for s, r in res_syn.items()}
c_rt = 2 * cost_bps / 1e4 * (1 + beta_true)
m_star = bertram_bands(kappa_true, sigma_true, c_rt)
theory = 2 * bertram_return_rate(m_star, kappa_true, sigma_true, c_rt) / (1 + beta_true)
print(f'Bertram theory: {100 * theory:.1f}% a year on gross capital; realised: {100 * realised["bertram"]:.1f}%')
print(f's-score realised: {100 * realised["sscore"]:.1f}% a year')

fig, ax = plt.subplots(figsize=(7, 3.2))
for s, r in res_syn.items():
    ax.plot((1 + r.committed).cumprod(), label=C.STRATEGY_LABELS[s], color=COLORS[s])
ax.set_ylabel('Wealth (start = 1)'); ax.legend()
savefig(fig, 'fig_e4_synthetic')

*M1: one paragraph: does the realised return match Bertram's prediction, and what explains the gap (estimation error from E1, discrete monitoring, window resets)?*

## 8. E5 headline backtest, US  (Owners: M1 engine, M4 strategies)  HEAVY
Rolling 12-month formation and 6-month trading windows, K = 20 pairs, 10 bp per leg. Design choices are frozen on 2010-2021; 2022-2026 is the untouched test block.

In [ ]:
if RUN_HEAVY:
    sel_us = run_selection('us', px_us, sectors('us'))     # slow once, then cached
    res_us = {s: simulate(s, px_us, sel_us, C.COST_BPS['us'], label=s) for s in STRATS}
    table1 = metrics_table(res_us.values(), bench_us).rename(index=C.STRATEGY_LABELS)
    table1.to_csv('results/table1_us.csv')
    split = pd.DataFrame({
        (C.STRATEGY_LABELS[s], blk): compute_metrics(r.committed[sl])
        for s, r in res_us.items()
        for blk, sl in [('2010-2021', slice(None, '2021-12-31')), ('2022-2026 test', slice(C.TEST_START, None))]
    }).T[['Ann. return %', 'Sharpe', 'Sharpe SE (Lo)', 'Max DD %']]
    split.to_csv('results/table1_split.csv')
    display(table1.round(2)); display(split.round(2))

In [ ]:
if RUN_HEAVY:
    fig, ax = plt.subplots(figsize=(9, 3.8))
    for s, r in res_us.items():
        ax.plot((1 + r.committed).cumprod(), lw=1.2, color=COLORS[s], label=C.STRATEGY_LABELS[s])
    ax.axvspan(pd.Timestamp(C.TEST_START), px_us.index[-1], color='grey', alpha=0.12, label='Test block')
    ax.set_ylabel('Wealth on committed capital (start = 1)'); ax.legend(fontsize=8)
    savefig(fig, 'fig_e5_cumulative')

*M4: interpret Table 1: which rule wins, is the Sharpe ratio significant (Lo SE, Newey-West t), and is the strategy market-neutral (beta)?*

## 9. E6 transaction costs  (Owner: M5)  HEAVY
Costs enter linearly, so the break-even cost per leg is where the annual return reaches zero.

In [ ]:
if RUN_HEAVY:
    cost_grid = [0, 5, 10, 20, 30]
    e6 = pd.DataFrame([dict(strategy=s, cost_bps=c, **compute_metrics(simulate(s, px_us, sel_us, c).committed))
                       for s in STRATS for c in cost_grid])
    e6.to_csv('results/e6_costs.csv', index=False)
    breakeven = {}
    fig, ax = plt.subplots(figsize=(6, 3.6))
    for s in STRATS:
        d = e6[e6.strategy == s]
        ax.plot(d.cost_bps, d['Sharpe'], marker='o', color=COLORS[s], label=C.STRATEGY_LABELS[s])
        slope, intercept = np.polyfit(d.cost_bps, d['Ann. return %'], 1)
        breakeven[C.STRATEGY_LABELS[s]] = -intercept / slope if slope < 0 else np.nan
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel('Cost per leg per transaction (bp)'); ax.set_ylabel('Sharpe ratio'); ax.legend(fontsize=8)
    savefig(fig, 'fig_e6_costs')
    display(pd.Series(breakeven, name='Break-even cost (bp per leg)').round(1))

## 10. E7 parameter sensitivity  (Owner: M5)  HEAVY
Reported as analysis only; the headline result keeps the paper thresholds.

In [ ]:
if RUN_HEAVY:
    entries, exits = [1.0, 1.25, 1.5, 2.0, 2.5], [0.0, 0.25, 0.5, 0.75]
    grid = pd.DataFrame(index=entries, columns=exits, dtype=float)
    for e in entries:
        for x in exits:
            p = dict(open_long=-e, open_short=e, close_long=-x, close_short=x)
            grid.loc[e, x] = compute_metrics(simulate('sscore', px_us, sel_us, C.COST_BPS['us'], params=p).committed)['Sharpe']
    grid.to_csv('results/e7_threshold_grid.csv')
    fig, ax = plt.subplots(figsize=(5, 3.6))
    im = ax.imshow(grid.values.astype(float), cmap='RdBu', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(exits)), exits); ax.set_yticks(range(len(entries)), entries)
    ax.set_xlabel('Exit |s|'); ax.set_ylabel('Entry |s|'); ax.grid(False)
    fig.colorbar(im, label='Sharpe ratio (S2, 10 bp)')
    savefig(fig, 'fig_e7_heatmap')

    robust = {}
    for f_days in [126, 252, 504]:
        sel_f = run_selection('us', px_us, sectors('us'), formation=f_days)
        robust[f'Formation {f_days} d'] = compute_metrics(simulate('sscore', px_us, sel_f, C.COST_BPS['us']).committed)
    for k in [5, 10, 20]:
        robust[f'K = {k}'] = compute_metrics(simulate('sscore', px_us, sel_us, C.COST_BPS['us'], k=k).committed)
    robust['Stop-loss |s| > 4'] = compute_metrics(simulate('sscore', px_us, sel_us, C.COST_BPS['us'], stop=4.0).committed)
    e7 = pd.DataFrame(robust).T[['Ann. return %', 'Sharpe', 'Sharpe SE (Lo)', 'Max DD %']]
    e7.to_csv('results/e7_robustness.csv'); display(e7.round(2))

## 11. E8 markets, sectors and sub-periods  (Owners: M5 tables, M2 India data)  HEAVY

In [ ]:
if RUN_HEAVY:
    sel_in = run_selection('india', px_in, sectors('india'))
    res_in = {s: simulate(s, px_in, sel_in, C.COST_BPS['india'], label=s) for s in STRATS}
    table_in = metrics_table(res_in.values(), bench_in).rename(index=C.STRATEGY_LABELS)
    table_in.to_csv('results/table_india.csv'); display(table_in.round(2))

    # Sharpe of S2 by sector (US): the sub-portfolio of that sector's selected pairs
    r = res_us['sscore']
    by_sector = {}
    for sec, cols in r.pairs.groupby('sector')['pair']:
        by_sector[sec] = compute_metrics(r.pair_returns[list(cols)].sum(axis=1))['Sharpe']
    display(pd.Series(by_sector, name='S2 Sharpe by sector').round(2))

    # Sub-periods, all strategies (US)
    sub = pd.DataFrame({name: {C.STRATEGY_LABELS[s]: compute_metrics(res.committed[a:b])['Sharpe']
                               for s, res in res_us.items()}
                        for name, (a, b) in C.SUB_PERIODS.items()})
    sub.to_csv('results/e8_subperiods.csv'); display(sub.round(2))

## 12. E9 do ADF-selected pairs stay cointegrated?  (Owner: M2)  HEAVY
For every selected pair we re-test the trading-window spread, built with the formation $\beta$, using ADF (no $\beta$ is re-estimated, so plain ADF critical values are correct here).

In [ ]:
if RUN_HEAVY:
    logp_us = np.log(px_us)
    rows = []
    for sel in sel_us:
        w = sel['window']
        for s in sel['coint']:
            x = (logp_us[s.a] - s.beta * logp_us[s.b] - s.mu).iloc[w.t0:w.t1].dropna().values
            if len(x) > 40:
                rows.append(dict(window=w.i, pair=f'w{w.i:02d}:{s.a}/{s.b}', eg_stat=s.eg_stat,
                                 half_life=s.half_life, still_coint=adf_test(x, maxlag=5)[1] < 0.05))
    e9 = pd.DataFrame(rows).merge(res_us['sscore'].pairs[['pair', 'pnl']], on='pair')
    print(f'Share of selected pairs still cointegrated out of sample: {100 * e9.still_coint.mean():.1f}%')
    e9['decile'] = pd.qcut(e9.eg_stat, 10, labels=False) + 1
    by_dec = e9.groupby('decile').pnl.mean() * 100
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.bar(by_dec.index, by_dec.values, color=COLORS['sscore'])
    ax.set_xlabel('Formation EG statistic decile (1 = most negative)'); ax.set_ylabel('Mean pair P&L per window (%)')
    savefig(fig, 'fig_e9_persistence')
    e9.to_csv('results/e9_persistence.csv', index=False)
    display(e9.groupby('still_coint').pnl.describe())

## 13. E10 Kalman-filter hedge ratio (stretch)  (Owner: M4)  HEAVY
State $(\mu_t,\beta_t)$ follows a random walk, observation $\ln P^A_t = \mu_t + \beta_t\ln P^B_t + e_t$ (Elliott et al. 2005; Chan 2013). Trade the standardised forecast error at $\pm 1$.

In [ ]:
if RUN_HEAVY:
    res_us['kalman'] = simulate('kalman', px_us, sel_us, C.COST_BPS['us'], label='kalman')
    display(metrics_table([res_us['kalman']], bench_us).rename(index=C.STRATEGY_LABELS).round(2))

## 14. Conclusions  (Owner: M5)
*Answer the three research questions in one short paragraph each, using the numbers above:*
1. How reliably do ADF and Engle-Granger find mean-reverting spreads (E2, E9)?
2. How well does the OU model describe them, and how accurate are its estimates (E1, E3, E4)?
3. Do OU-based rules beat the distance rule after costs, across markets and periods (E5-E8)?

*Limitations: survivorship bias (today's large caps), fixed hedge ratio within a window, daily closes only.*